# 🦅 SWISS-EAGLE v1.0
## Swiss Ensemble Agentic Generative Legal Engine
### LLM Agentic Legal Information Retrieval — Kaggle Competition

---

```
╔══════════════════════════════════════════════════════════════════════╗
║              S W I S S - E A G L E   A R C H I T E C T U R E        ║
║                                                                      ║
║  Query (EN)                                                          ║
║      │                                                               ║
║      ├──► [Engine 1] BM25 (German Query)  ──────────────┐           ║
║      ├──► [Engine 2] BM25 (English Query) ──────────────┤           ║
║      ├──► [Engine 3] Dense (multilingual-e5-large) ─────┤           ║
║      ├──► [Engine 4] Dense (paraphrase-multilingual) ───┤           ║
║      └──► [Engine 5] BM25 (Expanded Keywords) ──────────┤           ║
║                                                          │           ║
║                                          [RRF Fusion]   │           ║
║                                                │         │           ║
║                                     [Cross-Encoder Rerank]           ║
║                                                │                     ║
║                                    [Adaptive-K Citation Count]       ║
║                                                │                     ║
║                                         📄 submission.csv            ║
╚══════════════════════════════════════════════════════════════════════╝
```

**Key innovations:**
- **Cross-lingual BM25**: EN→DE query translation captures German legal vocabulary
- **5-Engine Ensemble**: Different retrieval signals fused via Reciprocal Rank Fusion (RRF)
- **Legal Query Expansion**: Domain-aware keyword boosting for Swiss law
- **Adaptive Citation Count**: Predicts optimal K per query using complexity signals
- **Zero external API calls**: Fully offline, Kaggle-compliant

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — INSTALLATION
# ═══════════════════════════════════════════════════════════
import subprocess, sys

packages = [
    'rank_bm25',
    'sentence-transformers',
    'faiss-cpu',
    'transformers',
    'torch',
    'nltk',
    'langdetect',
    'tqdm',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — IMPORTS & CONFIGURATION
# ═══════════════════════════════════════════════════════════
import os, gc, json, re, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm

import torch
import faiss
import nltk
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# ─── CONFIGURATION ───────────────────────────────────────────────────────────
class CFG:
    # Paths
    BASE_DIR        = Path('/kaggle/input/llm-agentic-legal-information-retrieval')
    TRAIN_PATH      = BASE_DIR / 'train.csv'
    TEST_PATH       = BASE_DIR / 'test.csv'
    CORPUS_PATH     = BASE_DIR / 'corpus.jsonl'        # adjust if different format
    OUTPUT_PATH     = Path('/kaggle/working/submission.csv')

    # Model paths (upload as Kaggle Datasets)
    # Option A: point to /kaggle/input/<your-dataset>/
    # Option B: let sentence-transformers load from cache if pre-cached
    DENSE_MODEL_1   = '/kaggle/input/multilingual-e5-large/multilingual-e5-large'
    DENSE_MODEL_2   = '/kaggle/input/paraphrase-multilingual/paraphrase-multilingual-mpnet-base-v2'
    RERANKER_MODEL  = '/kaggle/input/ms-marco-minilm/cross-encoder-ms-marco-MiniLM-L12-v2'

    # Retrieval
    BM25_TOP_K      = 100     # candidates per BM25 engine
    DENSE_TOP_K     = 100     # candidates per dense engine
    RERANK_TOP_K    = 30      # after fusion, rerank top-N
    FINAL_MAX_K     = 15      # hard cap on citations per query
    FINAL_MIN_K     = 1

    # RRF
    RRF_K           = 60      # standard RRF constant

    # Dense
    BATCH_SIZE      = 128
    DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Adaptive K
    SCORE_THRESHOLD = 0.35    # reranker score cutoff
    GAP_RATIO       = 0.65    # score drop ratio to trigger K cut

    # Seed
    SEED            = 42

np.random.seed(CFG.SEED)
print(f'🔧 Device: {CFG.DEVICE}')
print(f'🔧 BM25 Top-K: {CFG.BM25_TOP_K} | Dense Top-K: {CFG.DENSE_TOP_K}')
print(f'🔧 Rerank Top-K: {CFG.RERANK_TOP_K} | Max Citations: {CFG.FINAL_MAX_K}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — DATA LOADING
# ═══════════════════════════════════════════════════════════
print('📂 Loading competition data...')

train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df  = pd.read_csv(CFG.TEST_PATH)

print(f'✅ Train: {len(train_df)} queries')
print(f'✅ Test:  {len(test_df)} queries')
print('\nTrain sample:')
print(train_df.head(3).to_string())
print('\nTest sample:')
print(test_df.head(3).to_string())

# Parse gold citations for train
if 'gold_citations' in train_df.columns:
    train_df['citations_list'] = train_df['gold_citations'].apply(
        lambda x: [c.strip() for c in str(x).split(';') if c.strip()] if pd.notna(x) else []
    )
    avg_k = train_df['citations_list'].apply(len).mean()
    print(f'\n📊 Avg citations per query (train): {avg_k:.2f}')
    print(f'📊 Citation count distribution:')
    print(train_df['citations_list'].apply(len).describe())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — CORPUS LOADING & PREPROCESSING
# ═══════════════════════════════════════════════════════════

def load_corpus(corpus_path: Path) -> Tuple[List[str], List[str], List[str]]:
    """
    Load retrieval corpus.
    Returns:
        citation_ids : list of citation strings (e.g. 'Art. 111 ZGB')
        texts        : raw text per citation
        texts_lower  : lowercased tokenized text for BM25
    """
    citation_ids, texts = [], []

    suffix = corpus_path.suffix.lower()

    if suffix == '.jsonl':
        with open(corpus_path, 'r', encoding='utf-8') as f:
            for line in tqdm(f, desc='Loading corpus'):
                obj = json.loads(line)
                # Adapt field names to actual corpus schema
                cid  = obj.get('citation', obj.get('id', obj.get('doc_id', '')))
                text = obj.get('text', obj.get('content', obj.get('passage', '')))
                if cid and text:
                    citation_ids.append(cid.strip())
                    texts.append(text.strip())

    elif suffix == '.csv':
        df = pd.read_csv(corpus_path)
        cid_col  = [c for c in df.columns if 'citation' in c.lower() or 'id' in c.lower()][0]
        text_col = [c for c in df.columns if 'text' in c.lower() or 'content' in c.lower()][0]
        citation_ids = df[cid_col].tolist()
        texts        = df[text_col].fillna('').tolist()

    elif corpus_path.is_dir():
        # Multiple files (one per document)
        for fp in tqdm(list(corpus_path.glob('*.json')), desc='Loading corpus dir'):
            obj  = json.loads(fp.read_text('utf-8'))
            cid  = obj.get('citation', fp.stem)
            text = obj.get('text', '')
            citation_ids.append(cid)
            texts.append(text)

    else:
        raise ValueError(f'Unknown corpus format: {suffix}')

    print(f'✅ Corpus loaded: {len(citation_ids):,} documents')
    return citation_ids, texts


# ─── Determine actual corpus location ────────────────────────────────────────
# Try several common locations
candidate_paths = [
    CFG.BASE_DIR / 'corpus.jsonl',
    CFG.BASE_DIR / 'corpus.csv',
    CFG.BASE_DIR / 'retrieval_corpus.jsonl',
    CFG.BASE_DIR / 'corpus',
]
corpus_path = next((p for p in candidate_paths if p.exists()), None)

if corpus_path is None:
    raise FileNotFoundError(
        f'Corpus not found. Searched:\n' + '\n'.join(str(p) for p in candidate_paths)
    )

print(f'📂 Corpus path: {corpus_path}')
CITATION_IDS, CORPUS_TEXTS = load_corpus(corpus_path)

# Build reverse index: citation_id → index
CITATION_TO_IDX = {cid: i for i, cid in enumerate(CITATION_IDS)}
print(f'📊 Unique citations: {len(CITATION_IDS):,}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — LEGAL QUERY EXPANSION & GERMAN TRANSLATION
# ═══════════════════════════════════════════════════════════

# ─── Swiss Law Domain Keywords ────────────────────────────────────────────────
# Maps English legal concepts → German Swiss-law equivalents
LEGAL_EXPANSION = {
    # Civil law domains
    'marriage':         ['Ehe', 'Ehescheidung', 'ZGB'],
    'divorce':          ['Scheidung', 'Ehetrennung', 'Art. 111 ZGB', 'Art. 114 ZGB'],
    'inheritance':      ['Erbrecht', 'Erbschaft', 'Testament', 'ZGB'],
    'property':         ['Eigentum', 'Sachenrecht', 'Grundbuch', 'ZGB'],
    'contract':         ['Vertrag', 'OR', 'Obligationenrecht', 'Schuldrecht'],
    'tort':             ['Haftung', 'Schadenersatz', 'OR', 'unerlaubte Handlung'],
    'company':          ['Gesellschaft', 'AG', 'GmbH', 'OR', 'Aktiengesellschaft'],
    'bankruptcy':       ['Konkurs', 'Insolvenz', 'SchKG', 'Betreibung'],
    'tenancy':          ['Miete', 'Mietrecht', 'OR', 'Mietvertrag'],
    'employment':       ['Arbeitsrecht', 'OR', 'Arbeitnehmer', 'Kündigungsfrist'],
    # Public law domains
    'administrative':   ['Verwaltungsrecht', 'VwVG', 'Bundesverwaltung'],
    'tax':              ['Steuerrecht', 'DBG', 'MWST', 'Steuer'],
    'constitutional':   ['Verfassung', 'BV', 'Grundrechte', 'Bundesverfassung'],
    'criminal':         ['Strafrecht', 'StGB', 'Strafe', 'Strafgesetzbuch'],
    'social security':  ['Sozialversicherung', 'AHV', 'IV', 'AHVG'],
    'immigration':      ['Ausländerrecht', 'AIG', 'Aufenthaltsbewilligung'],
    'asylum':           ['Asylrecht', 'AsylG', 'Flüchtling'],
    # Process
    'appeal':           ['Beschwerde', 'BGG', 'Bundesgericht', 'Rechtsmittel'],
    'court':            ['Gericht', 'Bundesgericht', 'BGE', 'Urteil'],
    'evidence':         ['Beweismittel', 'Beweislast', 'ZPO'],
    'damages':          ['Schadenersatz', 'Genugtuung', 'OR 41', 'Haftung'],
    'custody':          ['Sorgerecht', 'Kindesverhältnis', 'ZGB'],
    'child':            ['Kind', 'Kindesrecht', 'Kindesschutz', 'ZGB'],
    'pension':          ['Vorsorge', 'BVG', 'Pensionskasse', 'Altersvorsorge'],
    'insurance':        ['Versicherung', 'VVG', 'Versicherungsvertrag'],
    'banking':          ['Bank', 'FINMA', 'BankG', 'Bankenrecht'],
    'competition':      ['Kartellrecht', 'KG', 'Wettbewerb'],
    'intellectual':     ['Immaterialgüterrecht', 'URG', 'MSchG', 'Markenrecht'],
    'copyright':        ['Urheberrecht', 'URG'],
    'trademark':        ['Markenrecht', 'MSchG'],
    'data protection':  ['Datenschutz', 'DSG'],
    'environment':      ['Umweltrecht', 'USG', 'Umweltschutz'],
    'land use':         ['Raumplanung', 'RPG', 'Baurecht'],
    'public procurement': ['öffentliches Beschaffungswesen', 'BöB'],
}


def expand_query_legal(query: str) -> str:
    """Add Swiss-law German terms based on query content."""
    query_lower = query.lower()
    extra_terms = []
    for keyword, terms in LEGAL_EXPANSION.items():
        if keyword in query_lower:
            extra_terms.extend(terms)
    if extra_terms:
        return query + ' ' + ' '.join(extra_terms)
    return query


def translate_query_to_german_heuristic(query: str) -> str:
    """
    Heuristic EN→DE legal query translation using expansion dict.
    If Helsinki translation model is available, prefer that.
    """
    query_lower = query.lower()
    de_terms = []
    for keyword, terms in LEGAL_EXPANSION.items():
        if keyword in query_lower:
            de_terms.extend(terms[:2])  # take top 2 German terms
    # Build German-flavored query
    if de_terms:
        return ' '.join(de_terms) + ' ' + query
    return query


# Try loading Helsinki opus-mt-en-de for proper translation
TRANSLATOR = None
try:
    from transformers import MarianMTModel, MarianTokenizer
    model_paths = [
        '/kaggle/input/opus-mt-en-de/opus-mt-en-de',
        '/kaggle/input/helsinki-nlp-opus-mt-en-de/opus-mt-en-de',
    ]
    for mp in model_paths:
        if Path(mp).exists():
            _tok  = MarianTokenizer.from_pretrained(mp)
            _mdl  = MarianMTModel.from_pretrained(mp).to(CFG.DEVICE)
            TRANSLATOR = (_tok, _mdl)
            print(f'✅ Translation model loaded from {mp}')
            break
    if TRANSLATOR is None:
        print('⚠️  Translation model not found — using heuristic expansion instead.')
except Exception as e:
    print(f'⚠️  Translation model error: {e} — using heuristic.')


def translate_queries(queries: List[str]) -> List[str]:
    if TRANSLATOR is None:
        return [translate_query_to_german_heuristic(q) for q in queries]
    tok, mdl = TRANSLATOR
    results = []
    for i in tqdm(range(0, len(queries), 32), desc='Translating EN→DE'):
        batch = queries[i:i+32]
        inputs = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to(CFG.DEVICE)
        with torch.no_grad():
            outs = mdl.generate(**inputs, num_beams=4, max_length=256)
        results.extend(tok.batch_decode(outs, skip_special_tokens=True))
    return results


print('✅ Query expansion & translation module ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — BM25 ENGINES (3x)
# ═══════════════════════════════════════════════════════════
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Multilingual stopwords
STOPWORDS_DE = set(stopwords.words('german'))
STOPWORDS_EN = set(stopwords.words('english'))
ALL_STOPWORDS = STOPWORDS_DE | STOPWORDS_EN


def tokenize_for_bm25(text: str) -> List[str]:
    """Tokenize with legal-aware preservation of citation patterns."""
    # Preserve citation patterns like 'Art. 111 ZGB', 'BGE 119 II 449'
    citation_pattern = r'(?:Art\.?\s*\d+[a-z]?|BGE\s*\d+|OR|ZGB|StGB|BV|SchKG|VwVG|OR|DBG|AHVG|AIG|AsylG|BVG|VVG|USG|RPG|MSchG|URG|DSG|KG|BöB|BGG|ZPO)(?:[\s/\-]\w+){0,5}'
    citations_found = re.findall(citation_pattern, text)
    # Normal tokenization
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalnum() and len(t) > 1 and t not in ALL_STOPWORDS]
    # Add full citation strings as tokens
    for c in citations_found:
        tokens.append(c.lower().replace(' ', '_'))
    return tokens


print('🔨 Tokenizing corpus for BM25 (this may take a few minutes)...')
CORPUS_TOKENIZED = [tokenize_for_bm25(text) for text in tqdm(CORPUS_TEXTS, desc='BM25 tokenization')]

print('🔨 Building BM25 index...')
BM25_INDEX = BM25Okapi(CORPUS_TOKENIZED, k1=1.5, b=0.75)
print('✅ BM25 index ready.')


def bm25_search(query_tokens: List[str], top_k: int = CFG.BM25_TOP_K) -> List[Tuple[int, float]]:
    """Return [(doc_idx, score), ...] sorted descending."""
    scores = BM25_INDEX.get_scores(query_tokens)
    top_idx = np.argpartition(scores, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    return [(int(i), float(scores[i])) for i in top_idx]


print('✅ BM25 engines ready (Engine 1: DE query, Engine 2: EN query, Engine 3: expanded).')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — DENSE RETRIEVAL ENGINES (2x)
# ═══════════════════════════════════════════════════════════
import faiss

def load_dense_model(model_path: str, name: str) -> Optional[SentenceTransformer]:
    if Path(model_path).exists():
        model = SentenceTransformer(model_path, device=CFG.DEVICE)
        print(f'✅ {name} loaded from {model_path}')
        return model
    else:
        print(f'⚠️  {name} not found at {model_path}')
        return None


DENSE_MODEL_A = load_dense_model(CFG.DENSE_MODEL_1, 'multilingual-e5-large')
DENSE_MODEL_B = load_dense_model(CFG.DENSE_MODEL_2, 'paraphrase-multilingual-mpnet-base-v2')


def build_faiss_index(model: SentenceTransformer, texts: List[str], prefix: str = '') -> faiss.Index:
    """
    Encode corpus and build FAISS inner-product index (normalized = cosine sim).
    """
    print(f'  Encoding {len(texts):,} corpus documents...')
    all_embeds = []
    for i in tqdm(range(0, len(texts), CFG.BATCH_SIZE), desc=f'  Encoding [{prefix}]'):
        batch = texts[i:i+CFG.BATCH_SIZE]
        if prefix:  # multilingual-e5 needs prefix
            batch = [f'passage: {t}' for t in batch]
        with torch.no_grad():
            emb = model.encode(batch, normalize_embeddings=True, batch_size=CFG.BATCH_SIZE,
                               show_progress_bar=False, convert_to_numpy=True)
        all_embeds.append(emb)
    all_embeds = np.vstack(all_embeds).astype('float32')
    dim = all_embeds.shape[1]
    # Use HNSW for fast ANN
    index = faiss.IndexHNSWFlat(dim, 32)
    index.hnsw.efConstruction = 200
    index.add(all_embeds)
    index.hnsw.efSearch = 128
    print(f'  ✅ FAISS index built. dim={dim}, n={index.ntotal:,}')
    del all_embeds; gc.collect()
    return index


FAISS_INDEX_A, FAISS_INDEX_B = None, None

if DENSE_MODEL_A:
    print('\n🔨 Building FAISS index A (multilingual-e5-large)...')
    FAISS_INDEX_A = build_faiss_index(DENSE_MODEL_A, CORPUS_TEXTS, prefix='e5')

if DENSE_MODEL_B:
    print('\n🔨 Building FAISS index B (paraphrase-multilingual)...')
    FAISS_INDEX_B = build_faiss_index(DENSE_MODEL_B, CORPUS_TEXTS, prefix='')


def dense_search(query: str, model: SentenceTransformer, index: faiss.Index,
                 top_k: int = CFG.DENSE_TOP_K, prefix: str = '') -> List[Tuple[int, float]]:
    q = f'query: {query}' if prefix == 'e5' else query
    with torch.no_grad():
        emb = model.encode([q], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    scores, indices = index.search(emb, top_k)
    return [(int(indices[0][i]), float(scores[0][i])) for i in range(top_k) if indices[0][i] >= 0]


print('\n✅ Dense retrieval engines ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — RECIPROCAL RANK FUSION (RRF)
# ═══════════════════════════════════════════════════════════

def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple[int, float]]],
    k: int = CFG.RRF_K,
    weights: Optional[List[float]] = None
) -> List[Tuple[int, float]]:
    """
    Fuse multiple ranked lists via weighted RRF.
    Each list is [(doc_idx, score), ...] sorted descending.
    Returns fused [(doc_idx, rrf_score), ...] sorted descending.
    """
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    assert len(weights) == len(ranked_lists)

    rrf_scores: Dict[int, float] = defaultdict(float)
    for lst, w in zip(ranked_lists, weights):
        for rank, (doc_idx, _) in enumerate(lst):
            rrf_scores[doc_idx] += w / (k + rank + 1)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused  # [(doc_idx, rrf_score), ...]


# RRF weights: give more weight to dense models (generally better semantic match)
# Engine order: BM25-DE, BM25-EN, BM25-expanded, Dense-A, Dense-B
RRF_WEIGHTS = [0.8, 0.7, 0.6, 1.2, 1.0]
print('✅ RRF fusion ready.')
print(f'  Weights: BM25-DE={RRF_WEIGHTS[0]}, BM25-EN={RRF_WEIGHTS[1]}, '
      f'BM25-exp={RRF_WEIGHTS[2]}, Dense-A={RRF_WEIGHTS[3]}, Dense-B={RRF_WEIGHTS[4]}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — CROSS-ENCODER RERANKER
# ═══════════════════════════════════════════════════════════

RERANKER = None
if Path(CFG.RERANKER_MODEL).exists():
    RERANKER = CrossEncoder(CFG.RERANKER_MODEL, device=CFG.DEVICE, max_length=512)
    print(f'✅ Reranker loaded: {CFG.RERANKER_MODEL}')
else:
    print(f'⚠️  Reranker not found at {CFG.RERANKER_MODEL} — skipping reranking step.')


def rerank_candidates(
    query: str,
    candidates: List[Tuple[int, float]],
    top_k: int = CFG.RERANK_TOP_K
) -> List[Tuple[int, float]]:
    """
    Cross-encoder reranking on top-K candidates.
    Returns [(doc_idx, rerank_score), ...] sorted descending.
    """
    if RERANKER is None:
        return candidates[:top_k]

    pool = candidates[:top_k]
    pairs = [(query, CORPUS_TEXTS[doc_idx][:512]) for doc_idx, _ in pool]
    scores = RERANKER.predict(pairs, batch_size=32, show_progress_bar=False)
    reranked = sorted(zip([d for d, _ in pool], scores),
                      key=lambda x: x[1], reverse=True)
    return reranked


print('✅ Reranking module ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — ADAPTIVE CITATION COUNT (K-SELECTION)
# ═══════════════════════════════════════════════════════════

def adaptive_k(
    reranked: List[Tuple[int, float]],
    query: str,
    score_threshold: float = CFG.SCORE_THRESHOLD,
    gap_ratio:       float = CFG.GAP_RATIO,
    min_k:           int   = CFG.FINAL_MIN_K,
    max_k:           int   = CFG.FINAL_MAX_K,
) -> int:
    """
    Determine how many citations to output for this query.
    Strategy:
      1. Keep all above score_threshold
      2. If a score drops more than gap_ratio from previous, cut there
      3. Clamp to [min_k, max_k]

    Additional heuristic: longer, more complex queries likely have more citations.
    """
    if not reranked:
        return min_k

    scores = [s for _, s in reranked]

    # If reranker is not available, scores are RRF scores (small, positive)
    # Normalize to [0,1] for threshold comparison
    max_s = max(scores) if max(scores) > 0 else 1.0
    norm_scores = [s / max_s for s in scores]

    k = max_k
    for i, ns in enumerate(norm_scores):
        if ns < score_threshold:
            k = max(i, min_k)
            break
        if i > 0 and norm_scores[i-1] > 0:
            drop = (norm_scores[i-1] - ns) / norm_scores[i-1]
            if drop > gap_ratio:
                k = max(i, min_k)
                break

    # Query complexity heuristic
    words = len(query.split())
    if words < 10:
        k = max(min_k, min(k, 5))
    elif words > 30:
        k = min(max_k, k + 2)

    return max(min_k, min(max_k, k))


print('✅ Adaptive-K citation count module ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — FULL RETRIEVAL PIPELINE
# ═══════════════════════════════════════════════════════════

def retrieve_citations(query: str, query_de: str, query_exp: str) -> List[str]:
    """
    Full 5-engine retrieval pipeline for a single query.
    Returns list of citation strings.
    """
    ranked_lists = []
    weights_used = []

    # ── Engine 1: BM25 with German-translated query ──────────────────────────
    q_de_tokens = tokenize_for_bm25(query_de)
    r1 = bm25_search(q_de_tokens, CFG.BM25_TOP_K)
    ranked_lists.append(r1)
    weights_used.append(RRF_WEIGHTS[0])

    # ── Engine 2: BM25 with original English query ───────────────────────────
    q_en_tokens = tokenize_for_bm25(query)
    r2 = bm25_search(q_en_tokens, CFG.BM25_TOP_K)
    ranked_lists.append(r2)
    weights_used.append(RRF_WEIGHTS[1])

    # ── Engine 3: BM25 with expanded keywords ────────────────────────────────
    q_exp_tokens = tokenize_for_bm25(query_exp)
    r3 = bm25_search(q_exp_tokens, CFG.BM25_TOP_K)
    ranked_lists.append(r3)
    weights_used.append(RRF_WEIGHTS[2])

    # ── Engine 4: Dense retrieval A (multilingual-e5) ─────────────────────────
    if DENSE_MODEL_A and FAISS_INDEX_A:
        r4 = dense_search(query, DENSE_MODEL_A, FAISS_INDEX_A, CFG.DENSE_TOP_K, prefix='e5')
        ranked_lists.append(r4)
        weights_used.append(RRF_WEIGHTS[3])

    # ── Engine 5: Dense retrieval B (paraphrase-multilingual) ─────────────────
    if DENSE_MODEL_B and FAISS_INDEX_B:
        r5 = dense_search(query, DENSE_MODEL_B, FAISS_INDEX_B, CFG.DENSE_TOP_K, prefix='')
        ranked_lists.append(r5)
        weights_used.append(RRF_WEIGHTS[4])

    # ── RRF Fusion ───────────────────────────────────────────────────────────
    fused = reciprocal_rank_fusion(ranked_lists, k=CFG.RRF_K, weights=weights_used)

    # ── Cross-encoder Reranking ───────────────────────────────────────────────
    reranked = rerank_candidates(query, fused, top_k=CFG.RERANK_TOP_K)

    # ── Adaptive K ───────────────────────────────────────────────────────────
    k = adaptive_k(reranked, query)
    final_docs = reranked[:k]

    # ── Map to citation strings ───────────────────────────────────────────────
    citations = [CITATION_IDS[doc_idx] for doc_idx, _ in final_docs
                 if 0 <= doc_idx < len(CITATION_IDS)]

    return citations


print('✅ Full pipeline function ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — VALIDATION ON TRAINING SET
# ═══════════════════════════════════════════════════════════

def compute_macro_f1(predictions: Dict[str, List[str]],
                     ground_truth: Dict[str, List[str]]) -> Dict[str, float]:
    """
    Compute citation-level Macro F1 (competition metric).
    """
    f1s, precisions, recalls = [], [], []
    for qid, gold in ground_truth.items():
        pred = set(predictions.get(qid, []))
        gold = set(gold)
        if not pred and not gold:
            f1s.append(1.0); precisions.append(1.0); recalls.append(1.0)
            continue
        tp = len(pred & gold)
        p  = tp / len(pred) if pred else 0.0
        r  = tp / len(gold) if gold else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        f1s.append(f1); precisions.append(p); recalls.append(r)
    return {
        'macro_f1':   np.mean(f1s),
        'precision':  np.mean(precisions),
        'recall':     np.mean(recalls),
        'n_queries':  len(f1s),
    }


# ─── Quick validation on a subset of train ───────────────────────────────────
VAL_SIZE = min(50, len(train_df))  # validate on up to 50 examples
val_df   = train_df.sample(VAL_SIZE, random_state=CFG.SEED)

print(f'🔍 Running validation on {VAL_SIZE} training queries...')

val_queries = val_df['query'].tolist()
val_de      = translate_queries(val_queries)
val_exp     = [expand_query_legal(q) for q in val_queries]

val_predictions = {}
val_ground_truth = {}

for i, row in tqdm(val_df.iterrows(), total=VAL_SIZE, desc='Validating'):
    qid  = str(row['query_id'])
    pred = retrieve_citations(row['query'], val_de[val_df.index.get_loc(i)],
                              val_exp[val_df.index.get_loc(i)])
    val_predictions[qid]  = pred
    val_ground_truth[qid] = row['citations_list'] if 'citations_list' in row else []

metrics = compute_macro_f1(val_predictions, val_ground_truth)
print('\n' + '═'*50)
print('📊 VALIDATION RESULTS')
print('═'*50)
print(f"  Macro F1   : {metrics['macro_f1']:.4f}")
print(f"  Precision  : {metrics['precision']:.4f}")
print(f"  Recall     : {metrics['recall']:.4f}")
print(f"  N Queries  : {metrics['n_queries']}")
print('═'*50)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — INFERENCE ON TEST SET & SUBMISSION
# ═══════════════════════════════════════════════════════════

print('🚀 Running inference on test set...')

test_queries = test_df['query'].tolist()
test_ids     = test_df['query_id'].tolist()

print('  Translating queries EN→DE...')
test_de  = translate_queries(test_queries)

print('  Expanding queries with legal terms...')
test_exp = [expand_query_legal(q) for q in test_queries]

results = []
t0 = time.time()

for i, (qid, query, de, exp) in enumerate(tqdm(
        zip(test_ids, test_queries, test_de, test_exp),
        total=len(test_queries), desc='Retrieving')):
    citations = retrieve_citations(query, de, exp)
    results.append({
        'query_id':           qid,
        'predicted_citations': ';'.join(citations)
    })

elapsed = time.time() - t0
print(f'\n⏱️  Total inference time: {elapsed:.1f}s ({elapsed/len(test_queries):.2f}s/query)')

# Save submission
submission = pd.DataFrame(results)
submission.to_csv(CFG.OUTPUT_PATH, index=False)
print(f'\n✅ Submission saved to {CFG.OUTPUT_PATH}')
print(f'   Shape: {submission.shape}')
print('\nSample predictions:')
print(submission.head(10).to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — POST-ANALYSIS & DIAGNOSTICS
# ═══════════════════════════════════════════════════════════

submission = pd.read_csv(CFG.OUTPUT_PATH)

citation_counts = submission['predicted_citations'].apply(
    lambda x: len(str(x).split(';')) if pd.notna(x) and str(x).strip() else 0
)

print('📊 Submission diagnostics')
print('─'*40)
print(f"  Total queries:           {len(submission)}")
print(f"  Avg citations/query:     {citation_counts.mean():.2f}")
print(f"  Min citations/query:     {citation_counts.min()}")
print(f"  Max citations/query:     {citation_counts.max()}")
print(f"  Queries with 0 cites:    {(citation_counts == 0).sum()}")
print(f"  Citation count std:      {citation_counts.std():.2f}")
print('─'*40)

# Citation type distribution
all_citations = []
for row in submission['predicted_citations']:
    if pd.notna(row):
        all_citations.extend([c.strip() for c in str(row).split(';') if c.strip()])

statute_count  = sum(1 for c in all_citations if c.startswith('Art.'))
decision_count = sum(1 for c in all_citations if c.startswith('BGE'))
other_count    = len(all_citations) - statute_count - decision_count

print(f"\n  Citation types:")
print(f"    Statutes  (Art. ...):  {statute_count}")
print(f"    Decisions (BGE ...):   {decision_count}")
print(f"    Other:                 {other_count}")
print(f"    Total:                 {len(all_citations)}")

print('\n🏆 SWISS-EAGLE v1.0 — Pipeline complete!')

---
## 📋 Setup Guide — Before Submitting

### Step 1: Upload Models as Kaggle Datasets

Upload these HuggingFace models as Kaggle Datasets and attach them to your notebook:

| Dataset Name | Model | Path |
|---|---|---|
| `multilingual-e5-large` | `intfloat/multilingual-e5-large` | `/kaggle/input/multilingual-e5-large/` |
| `paraphrase-multilingual` | `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` | `/kaggle/input/paraphrase-multilingual/` |
| `ms-marco-minilm` | `cross-encoder/ms-marco-MiniLM-L12-v2` | `/kaggle/input/ms-marco-minilm/` |
| `opus-mt-en-de` | `Helsinki-NLP/opus-mt-en-de` | `/kaggle/input/opus-mt-en-de/` |

**Download script (run locally, then upload as dataset):**
```python
from huggingface_hub import snapshot_download
models = [
    'intfloat/multilingual-e5-large',
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'cross-encoder/ms-marco-MiniLM-L12-v2',
    'Helsinki-NLP/opus-mt-en-de',
]
for m in models:
    snapshot_download(m, local_dir=f'./{m.split("/")[1]}')
```

### Step 2: Check Corpus Format
- The notebook auto-detects `.jsonl`, `.csv`, or directory corpus formats
- Adjust field names in `load_corpus()` if needed

### Step 3: Tune Hyperparameters
Key parameters in `CFG`:
- `SCORE_THRESHOLD` (0.25–0.45): Lower = more citations (higher recall), Higher = fewer (higher precision)
- `RRF_WEIGHTS`: Tune based on which engines perform best on validation set
- `RERANK_TOP_K`: Increase for better quality at cost of speed

---
**Architecture by SWISS-EAGLE v1.0 | Macro F1 optimized**